# Chapter 7 - Lab 5: <font color='blue'>Parallel Multi-Company Analysis</font>

**<font color='purple'>Goal</font>**:
Take the 4-agent pipeline from Lab 1 and run it for **multiple companies concurrently** using `asyncio.gather`. This is the **Parallelization** architectural style (§2.5) — independent tasks distributed across concurrent pipeline instances.

By the end of the lab you will compare wall-clock time between sequential and parallel runs across a five-stock portfolio.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex AgentWorkflow** (same as Lab 1).
* **`asyncio.gather`** — schedule concurrent pipeline instances.
* **OpenAI** `gpt-4o`.
* **Stub data** so the lab is self-contained and reproducible.

Prerequisite: read Lab 1 first.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 7 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%207/common.py',
        'common.py',
    )

from common import (
    PROFITABILITY_THRESHOLDS, LIQUIDITY_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)

print('Common helpers loaded.')

## 4. Re-create the Lab 1 pipeline as a callable

Wrap the entire 4-agent pipeline in a single async function `analyze(ticker)` so we can schedule multiple calls in parallel. The internal definitions are identical to Lab 1 — abbreviated here for brevity.

In [ ]:
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)


async def get_fundamentals(ctx: Context, ticker: str) -> str:
    data = stub_lookup(ticker)
    state = await ctx.get('state')
    state['ticker'] = ticker
    state['ratios'] = {**data['profitability'], **data['liquidity']}
    await ctx.set('state', state)
    return 'fundamentals ok'

async def get_profitability_ratios(ctx: Context) -> str:
    state = await ctx.get('state')
    state['profitability_ratios'] = {k: state['ratios'][k] for k in PROFITABILITY_THRESHOLDS}
    state['threshold_profitability_comments'] = format_threshold_prompt(PROFITABILITY_THRESHOLDS, 'Profitability')
    await ctx.set('state', state)
    return 'profitability ok'

async def get_liquidity_ratios(ctx: Context) -> str:
    state = await ctx.get('state')
    state['liquidity_ratios'] = {k: state['ratios'][k] for k in LIQUIDITY_THRESHOLDS}
    state['threshold_liquidity_comments'] = format_threshold_prompt(LIQUIDITY_THRESHOLDS, 'Liquidity')
    await ctx.set('state', state)
    return 'liquidity ok'

async def get_overall_comments(ctx: Context) -> str:
    state = await ctx.get('state')
    state['overall_comments'] = 'synthesis complete'
    await ctx.set('state', state)
    return 'overall ok'

In [ ]:
def build_pipeline() -> AgentWorkflow:
    fa = FunctionAgent(name='FundamentalAgent', description='', system_prompt='Hand off to ProfitabilityAgent.', llm=llm, tools=[get_fundamentals], can_handoff_to=['ProfitabilityAgent'])
    pa = FunctionAgent(name='ProfitabilityAgent', description='', system_prompt='Hand off to LiquidityAgent.', llm=llm, tools=[get_profitability_ratios], can_handoff_to=['LiquidityAgent'])
    la = FunctionAgent(name='LiquidityAgent', description='', system_prompt='Hand off to SupervisorAgent.', llm=llm, tools=[get_liquidity_ratios], can_handoff_to=['SupervisorAgent'])
    sa = FunctionAgent(name='SupervisorAgent', description='', system_prompt='Synthesise verdict.', llm=llm, tools=[get_overall_comments], can_handoff_to=['FundamentalAgent'])
    return AgentWorkflow(
        agents=[fa, pa, la, sa],
        root_agent='FundamentalAgent',
        initial_state={'ratios': {}, 'ticker': '',
                       'profitability_ratios': {}, 'liquidity_ratios': {},
                       'threshold_profitability_comments': None,
                       'threshold_liquidity_comments': None,
                       'overall_comments': None},
    )


async def analyze(ticker: str) -> str:
    pipeline = build_pipeline()
    handler = pipeline.run(user_msg=f'Analyse {ticker} financial health.')
    return str(await handler)

## 5. Run sequentially as a baseline

Time the sequential run across the portfolio so we know what speedup to expect from parallelisation.

In [ ]:
import asyncio, time

PORTFOLIO = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'JPM']

async def sequential_run():
    results = {}
    for t in PORTFOLIO:
        results[t] = await analyze(t)
    return results

start = time.time()
sequential_results = await sequential_run()
sequential_elapsed = time.time() - start
print(f'Sequential: {sequential_elapsed:.1f}s for {len(PORTFOLIO)} companies.')

## 6. Run in parallel with `asyncio.gather`

Each ticker gets its own pipeline instance. `asyncio.gather` schedules them all on the same event loop. Because most of the wall-clock time is spent waiting on OpenAI responses, the concurrent runs overlap almost completely.

In [ ]:
async def parallel_run():
    tasks = [analyze(t) for t in PORTFOLIO]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return dict(zip(PORTFOLIO, results))

start = time.time()
parallel_results = await parallel_run()
parallel_elapsed = time.time() - start
print(f'Parallel:   {parallel_elapsed:.1f}s for {len(PORTFOLIO)} companies.')
print(f'Speedup:    {sequential_elapsed / parallel_elapsed:.1f}x')

## 7. Compare results

Both runs should produce equivalent verdicts (modulo small LLM nondeterminism). Print the first 200 characters of each to confirm.

In [ ]:
print(f'{"Ticker":<8} {"Sequential output (200ch)":<60} {"Parallel output (200ch)"}')
for t in PORTFOLIO:
    seq = sequential_results[t][:60].replace('\n', ' ')
    par = parallel_results[t]
    if isinstance(par, Exception):
        par_str = f'ERROR: {par}'
    else:
        par_str = str(par)[:60].replace('\n', ' ')
    print(f'{t:<8} {seq:<60} {par_str}')

## 8. Results

Same analysis pipeline, run N times in parallel. **What to notice about parallelisation:**

* **The speedup is approximately the number of companies**, because each pipeline is dominated   by network-bound LLM calls. CPU-bound work would not parallelise as well in a single Python process.
* **Independence is the prerequisite.** This works because each company's analysis is   self-contained. If they shared state, you would need locks (rarely worth the complexity).
* **Failure isolation.** `return_exceptions=True` keeps one company's failure from killing the whole run.
* **Rate limits matter.** N parallel pipelines mean N× tokens per second to the provider. Stay   under your tier's TPM or you will see throttling.

Combining patterns: in production you might stack Lab 4 (evaluator-optimizer) inside Lab 5 (parallel) — every pipeline instance self-corrects, and all instances run concurrently. The chapter discusses how to compose styles like this in §3.2.